# Meeting Assistant — Pipeline

Ejecuta las celdas en orden:
1. **Config + conectividad** — define rutas, contexto y valida la API
2. **Listar transcripciones** — muestra archivos pendientes y ya procesados
3. **Pipeline** — procesa cada transcripción y genera outputs
4. **Resumen** — lista los resultados generados

In [1]:
import sys, json
from pathlib import Path

sys.path.append(str(Path("..") / "src"))

from meeting_assistant import (
    make_client,
    load_transcript,
    list_transcripts,
    preprocess_transcript,
    extract_structured,
    estimate_requests,
    load_cached,
    to_markdown,
)

# ── Rutas ──────────────────────────────────────────────────────────────────
ENV_PATH  = "../.env"
TRANS_DIR = Path("../transcriptions")
OUT_DIR   = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

# ── Contexto de la reunión (opcional) ─────────────────────────────────────
# Proporciona contexto para que el modelo entienda el tipo de reunión.
# Deja vacío ("") para dejar que el modelo lo infiera automáticamente.
#
# Ejemplos:
#   "Reunión técnica de desarrollo backend. Revisión de mejoras en la plataforma."
#   "Reunión estratégica de ventas Q1. Equipo comercial Geoperuvian."
#   "Weekly del equipo de dirección. Seguimiento de proyectos activos."
MEETING_CONTEXT = "La reunión trató sobre implementar mejoras en la plataforma Meta Mining Media con herramientas populares usadas en el rubro de marketing como COR y Slack. El foco fue identificar que funcionalidades son necesarias en la plataforma de MMM para implementarlas y tener una plataforma robusta todo en uno"

# ── Conectar y hacer smoke test ─────────────────────────────────────────────
client, model, km = make_client(ENV_PATH)
resp = client.models.generate_content(model=model, contents="Di solo OK")
print(f"Modelo  : {model}")
print(f"Conexión: {resp.text.strip()}")
if MEETING_CONTEXT:
    print(f"Contexto: {MEETING_CONTEXT}")
else:
    print("Contexto: (no especificado — el modelo lo infiere)")

Modelo  : models/gemini-2.5-flash
Conexión: OK
Contexto: La reunión trató sobre implementar mejoras en la plataforma Meta Mining Media con herramientas populares usadas en el rubro de marketing como COR y Slack. El foco fue identificar que funcionalidades son necesarias en la plataforma de MMM para implementarlas y tener una plataforma robusta todo en uno


In [2]:
# ── Listar transcripciones disponibles ────────────────────────────────────
files = list_transcripts(TRANS_DIR)
print(f"Transcripciones encontradas: {len(files)}\n")
for f in files:
    out_json = OUT_DIR / f"{f.stem}_structured.json"
    status = "✅ ya procesado" if out_json.exists() else "⏳ pendiente"
    print(f"  · {f.name}  [{status}]")

Transcripciones encontradas: 1

  · plataforma-MMM_260203.docx  [⏳ pendiente]


In [3]:
# ── Pipeline principal ─────────────────────────────────────────────────────
# Para reprocesar un archivo ya procesado:
#   1. Elimina su *_structured.json de outputs/
#   2. Vuelve a ejecutar esta celda

results = []

for f in files:
    out_json = OUT_DIR / f"{f.stem}_structured.json"
    out_md   = OUT_DIR / f"{f.stem}.md"

    print(f"\n{'─'*55}")
    print(f"📄 {f.name}")

    # Caché: skip si ya existe el JSON estructurado
    if load_cached(out_json) is not None:
        print(f"   [SKIP] Ya procesado → {out_json.name}")
        results.append({"file": f.name, "status": "SKIP"})
        continue

    try:
        # 1. Cargar y preprocesar
        text = load_transcript(f)
        text = preprocess_transcript(text, verbose=True)

        # 2. Estimar requests
        plan = estimate_requests(text, max_chars_single_shot=45000, chunk_chars=30000)
        print(f"   Modo: {plan['mode']} · Requests estimados: {plan['requests_generate']}")

        # 3. Extraer JSON estructurado
        data = extract_structured(
            client, model, km, text,
            meeting_context=MEETING_CONTEXT
        )

        # 4. Guardar outputs
        out_json.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
        out_md.write_text(to_markdown(data), encoding="utf-8")

        print(f"   ✅ Guardado:")
        print(f"      · {out_json.name}")
        print(f"      · {out_md.name}")
        results.append({"file": f.name, "status": "OK", "json": out_json.name, "md": out_md.name})

    except Exception as e:
        print(f"   ❌ ERROR: {e}")
        results.append({"file": f.name, "status": "ERROR", "error": str(e)})

print(f"\n{'─'*55}")
print("Pipeline finalizado.")


───────────────────────────────────────────────────────
📄 plataforma-MMM_260203.docx
  Preprocesamiento: 34,185 → 33,623 chars  (1.6% reducción)
   Modo: single · Requests estimados: 1
   ✅ Guardado:
      · plataforma-MMM_260203_structured.json
      · plataforma-MMM_260203.md

───────────────────────────────────────────────────────
Pipeline finalizado.


In [4]:
# ── Resumen de outputs generados ──────────────────────────────────────────
json_files = sorted(OUT_DIR.glob("*_structured.json"))
if not json_files:
    print("No hay outputs generados aún.")
else:
    print(f"Outputs en {OUT_DIR}:\n")
    for p in json_files:
        try:
            d     = json.loads(p.read_text(encoding="utf-8"))
            title = d.get("meeting_title", "?")
            date  = d.get("date", "?")
            mode  = d.get("_meta", {}).get("mode", "?")
            reqs  = d.get("_meta", {}).get("requests_generate", "?")
            topics = [t.get("name", "") for t in d.get("topics", [])]
            n_acts = len(d.get("action_items", []))
            print(f"  · {title} ({date})")
            print(f"    Modo: {mode} · Requests: {reqs} · Tareas: {n_acts}")
            if topics:
                print(f"    Temas: {', '.join(topics)}")
            print(f"    Archivo: {p.name}")
        except Exception as e:
            print(f"  · {p.name} — error: {e}")

Outputs en ..\outputs:

  · Mejoras y Automatización de la Plataforma Meta Mining Media (MMM) (2 de marzo de 2026)
    Modo: single · Requests: 1 · Tareas: 4
    Temas: Funcionalidades de la plataforma Core, Integración de IA para revisión de piezas y contenido, Integración de Slack para comunicación, Roles y jerarquías en la plataforma, Automatización de procesos de marketing con IA, Generación de entregables y assets, Optimización de la comunicación comercial
    Archivo: plataforma-MMM_260203_structured.json
